# SBC

Detailed workflow for this experiment. For catalogue inputs and dependencies, see [data](../../../docs/data.ipynb). Enable `LG_RUN_RESEARCH_WORKFLOW=1` before running the calculation cells.

In [ ]:
import os
RUN_RESEARCH_WORKFLOW = os.environ.get("LG_RUN_RESEARCH_WORKFLOW") == "1"
if not RUN_RESEARCH_WORKFLOW:
    raise RuntimeError("Optional workflow disabled. Read notebooks/92_reproduce_saved_analyses.ipynb and set LG_RUN_RESEARCH_WORKFLOW=1.")

In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file())
os.environ.setdefault("LG_PROJECT", str(ROOT / "research"))
PROJECT = Path(os.environ["LG_PROJECT"]).expanduser().resolve()
os.chdir(PROJECT)

def design_path(logical_name):
    if logical_name == "FINAL_EXPERIMENT_DESIGN_2026-10-01.md":
        return PROJECT / "out/final_stage_audit/portability_v3/original_design.txt"
    return PROJECT / logical_name

### Corrected catalogue selection
Corrected TNG selection and results use separate v3 paths. Build the sample with
`python -X utf8 -B TNG300/rebuild_comparison.py`, then run `inference_npe.ipynb`
before dependent analyses in the original Roihu training environment.
The main FLAMINGO models are reused; their LG samples are refreshed after converting
physical observed radial velocity to peculiar velocity. The scalar affine MAF is
conditional Gaussian in log10 mass before posterior support restriction. Marginalizing
uncertain features can produce a non-Gaussian mixture. Calibration reports actual
central-interval coverage and descriptive discrete ranks; beta=0.6745 is only a
Gaussian reference. Historical source and outputs are preserved in
`review_backups/20260906_210956`; the working outputs must be regenerated.


In [ ]:
import os, sys
from pathlib import Path
PROJECT = Path(os.environ.get("LG_PROJECT", r"<research-project>"))
sys.path.insert(0, str(PROJECT / "analogues"))
from review_fixes import (RUN_VERSION, MODEL_DESCRIPTION, H0_BY_SIM, configure_run,
    feature_cache_dir, lg_observation, lg_to_peculiar, draw_noisy_feature,
    sample_lg_point, sample_lg_marginalized, calibration_summary, tng_volume_mpc3)
LEGACY_RESULTS_DIR = PROJECT / "pickles_v2"
FEATURE_CACHE_DIR = feature_cache_dir(PROJECT)

from review_fixes import require_legacy_environment
require_legacy_environment(PROJECT)


In [ ]:
import os, sys
from pathlib import Path
PROJECT  = Path(os.environ.get('LG_PROJECT', r'<research-project>'))
TNG_BASE = os.environ.get('TNG_BASE', r'<TNG_BASE>')
sys.path.insert(0, str(PROJECT / 'analogues'))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import functools
from pathlib import Path
import torch
from scipy.stats import uniform, kstest
import sys
sys.path.insert(0, str(PROJECT / "analogues"))
from splits import pair_groups, group_split

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
results_dir = configure_run(PROJECT) 
results_dir.mkdir(parents=True, exist_ok=True)

def load_checkpoint_any_device(path, device):
    original_torch_load = torch.load
    torch.load = functools.partial(original_torch_load, map_location=device)
    try:
        with open(path, "rb") as f:
            obj = pickle.load(f)
    finally:
        torch.load = original_torch_load
    return obj

flam_ckpt = load_checkpoint_any_device(results_dir / "flamingo_npe_checkpoint.pkl", device)
flam_posteriors = flam_ckpt["posteriors"]
flam_scalers    = flam_ckpt["scalers"]

tng_ckpt = load_checkpoint_any_device(results_dir / "tng_npe_checkpoint.pkl", device)
tng_posteriors = tng_ckpt["posteriors"]
tng_scalers    = tng_ckpt["scalers"]

BLOCKS = {
    "core_dynamics":   ["separation", "vr", "vt", "log_mass_ratio"],
    "core_chemistry":  ["separation", "vr", "vt", "log_mass_ratio", "met_diff"],
    "core_morphology": ["separation", "vr", "vt", "log_mass_ratio",
                        "sigma_mean", "sigma_diff", "kappa_mean", "kappa_diff"],
    "core_photometry": ["separation", "vr", "vt", "log_mass_ratio", "gr_mean", "gr_diff"],
    "all_features":    ["separation", "vr", "vt", "log_mass_ratio",
                        "sigma_mean", "sigma_diff", "kappa_mean", "kappa_diff",
                        "met_diff", "gr_mean", "gr_diff"],
}
print(f"Loaded {len(flam_posteriors)} FLAMINGO combos, {len(tng_posteriors)} TNG blocks.")

In [ ]:
import pickle as pkl
from sklearn.model_selection import train_test_split

# FLAMINGO raw data
cache_dir = PROJECT / "analogues/cache/run_001_mstar2e10"   

with open(cache_dir / "flamingo_sample.pkl", "rb") as f:
    sample = pkl.load(f)
with open(cache_dir / "flamingo_pairs.pkl", "rb") as f:
    pairs = pkl.load(f)
with open(FEATURE_CACHE_DIR / "metallicity_log_zscore.pkl", "rb") as f:
    met_transform = pkl.load(f)

def pair_features_flamingo(sample, pairs, met_transform):
    pi, pj = pairs["i"], pairs["j"]
    gr_i, gr_j = sample["g_r"][pi], sample["g_r"][pj]
    sigma_i, sigma_j = sample["vel_disp"][pi], sample["vel_disp"][pj]
    kap_i, kap_j = sample["kappa_rot"][pi], sample["kappa_rot"][pj]
    met_i = met_transform["flamingo_met_dex"][pi]
    met_j = met_transform["flamingo_met_dex"][pj]
    return pd.DataFrame({
        "gr_mean": (gr_i+gr_j)/2, "gr_diff": np.abs(gr_i-gr_j),
        "met_mean": (met_i+met_j)/2, "met_diff": np.abs(met_i-met_j),
        "sigma_mean": (sigma_i+sigma_j)/2, "sigma_diff": np.abs(sigma_i-sigma_j),
        "kappa_mean": (kap_i+kap_j)/2, "kappa_diff": np.abs(kap_i-kap_j),
        "separation": pairs["separation_kpc"], "vr": pairs["vr"], "vt": pairs["vt"],
        "log_mass_ratio": pairs["log_mass_ratio"], "log_m200c": pairs["log_tot_virial_mass"],
        "host_i": sample["host_top_index"][pi],
        "host_j": sample["host_top_index"][pj],
    })

flam_df = pair_features_flamingo(sample, pairs, met_transform)
flam_df["same_host"] = pairs["have_same_host"].astype(bool)

# TNG raw data
tng_dir = PROJECT / "analogues/tng_for_comparison_v3"  

with open(FEATURE_CACHE_DIR / "tng_sigma_star.pkl", "rb") as f:
    tng_sigma_star = pkl.load(f)
with open(FEATURE_CACHE_DIR / "tng_kappa_corot.pkl", "rb") as f:
    kappa_cache = pkl.load(f)

keep_idx    = np.load(tng_dir / "keep_idx.npy")
tng_pairs_i = np.load(tng_dir / "pairs_i.npy")
tng_pairs_j = np.load(tng_dir / "pairs_j.npy")
tng_gr_sample   = np.load(tng_dir / "g_r.npy")
tng_met_sample  = np.load(tng_dir / "metallicity.npy")
tng_separation  = np.load(tng_dir / "separation.npy")
tng_vr          = np.load(tng_dir / "vr.npy")
tng_vt          = np.load(tng_dir / "vt.npy")
tng_log_m200c   = np.load(tng_dir / "log_m200c.npy")
tng_log_mass_ratio = np.load(tng_dir / "log_mass_ratio.npy")
tng_same_host   = np.load(tng_dir / "have_same_host.npy").astype(bool)
tng_grnr = np.load(tng_dir / "grnr.npy")

tng_pairs_i_global = keep_idx[tng_pairs_i]
tng_pairs_j_global = keep_idx[tng_pairs_j]
tng_sigma_i = np.array([tng_sigma_star.get(int(x), np.nan) for x in tng_pairs_i_global])
tng_sigma_j = np.array([tng_sigma_star.get(int(x), np.nan) for x in tng_pairs_j_global])
tng_kappa_i = np.array([kappa_cache.get(int(x), np.nan) for x in tng_pairs_i_global])
tng_kappa_j = np.array([kappa_cache.get(int(x), np.nan) for x in tng_pairs_j_global])

def pair_features_tng(tng_pairs_i, tng_pairs_j, tng_gr_sample, tng_met_sample,
                       tng_sigma_i, tng_sigma_j, tng_kappa_i, tng_kappa_j,
                       tng_separation, tng_vr, tng_vt, tng_log_mass_ratio,
                       tng_log_m200c, met_transform):
    gr_i, gr_j = tng_gr_sample[tng_pairs_i], tng_gr_sample[tng_pairs_j]
    met_raw_i = tng_met_sample[tng_pairs_i]
    met_raw_j = tng_met_sample[tng_pairs_j]
    met_i = met_transform["tng_met_dex"][tng_pairs_i]
    met_j = met_transform["tng_met_dex"][tng_pairs_j]
    return pd.DataFrame({
        "gr_mean": (gr_i+gr_j)/2, "gr_diff": np.abs(gr_i-gr_j),
        "met_mean": (met_i+met_j)/2, "met_diff": np.abs(met_i-met_j),
        "sigma_mean": (tng_sigma_i+tng_sigma_j)/2, "sigma_diff": np.abs(tng_sigma_i-tng_sigma_j),
        "kappa_mean": (tng_kappa_i+tng_kappa_j)/2, "kappa_diff": np.abs(tng_kappa_i-tng_kappa_j),
        "separation": tng_separation, "vr": tng_vr, "vt": tng_vt,
        "log_mass_ratio": tng_log_mass_ratio, "log_m200c": tng_log_m200c,
        "host_i": tng_grnr[tng_pairs_i],
        "host_j": tng_grnr[tng_pairs_j],
    })

tng_df = pair_features_tng(
    tng_pairs_i, tng_pairs_j, tng_gr_sample, tng_met_sample,
    tng_sigma_i, tng_sigma_j, tng_kappa_i, tng_kappa_j,
    tng_separation, tng_vr, tng_vt, tng_log_mass_ratio,
    tng_log_m200c, met_transform
)
tng_df["same_host"] = tng_same_host

# clean splits matches everything used to train the checkpointed models
ALL_FEATURES = BLOCKS["all_features"]
flam_df_clean = flam_df.dropna(subset=ALL_FEATURES + ["log_m200c"])
tng_df_clean  = tng_df.dropna(subset=ALL_FEATURES + ["log_m200c"])

flam_same = flam_df_clean[flam_df_clean["same_host"]].copy()
flam_diff = flam_df_clean[~flam_df_clean["same_host"]].copy()

def get_test_split(df, feats, seed=42, test_size=0.15):
    cols  = feats + ["log_m200c", "host_i", "host_j"]
    valid = df[cols].dropna(subset=feats + ["log_m200c"]).reset_index(drop=True)
    _, te = group_split(valid, test_size=test_size, seed=seed)
    return valid.iloc[te]

DATASETS = {"full": flam_df_clean, "same_host": flam_same, "diff_host": flam_diff}

print(f"FLAMINGO: {len(flam_df_clean):,} (same-host {len(flam_same):,}, diff-host {len(flam_diff):,})")
print(f"TNG: {len(tng_df_clean):,}")

In [ ]:
cache_dir = PROJECT / "analogues/cache/run_001_mstar2e10"   
tng_dir   = PROJECT / "analogues/tng_for_comparison_v3"

with open(cache_dir / "flamingo_sample.pkl", "rb") as f:
    sample = pkl.load(f)
with open(cache_dir / "flamingo_pairs.pkl", "rb") as f:
    pairs = pkl.load(f)
with open(FEATURE_CACHE_DIR / "metallicity_log_zscore.pkl", "rb") as f:
    met_transform = pkl.load(f)

def pair_features_flamingo(sample, pairs, met_transform):
    pi, pj = pairs["i"], pairs["j"]
    gr_i, gr_j = sample["g_r"][pi], sample["g_r"][pj]
    sigma_i, sigma_j = sample["vel_disp"][pi], sample["vel_disp"][pj]
    kap_i, kap_j = sample["kappa_rot"][pi], sample["kappa_rot"][pj]
    met_i = met_transform["flamingo_met_dex"][pi]
    met_j = met_transform["flamingo_met_dex"][pj]
    return pd.DataFrame({
        "gr_mean": (gr_i+gr_j)/2, "gr_diff": np.abs(gr_i-gr_j),
        "met_mean": (met_i+met_j)/2, "met_diff": np.abs(met_i-met_j),
        "sigma_mean": (sigma_i+sigma_j)/2, "sigma_diff": np.abs(sigma_i-sigma_j),
        "kappa_mean": (kap_i+kap_j)/2, "kappa_diff": np.abs(kap_i-kap_j),
        "separation": pairs["separation_kpc"], "vr": pairs["vr"], "vt": pairs["vt"],
        "log_mass_ratio": pairs["log_mass_ratio"], "log_m200c": pairs["log_tot_virial_mass"],
        "host_i": sample["host_top_index"][pi],
        "host_j": sample["host_top_index"][pj],
    })

flam_df = pair_features_flamingo(sample, pairs, met_transform)
flam_df["same_host"] = pairs["have_same_host"].astype(bool)
ALL_FEATURES = BLOCKS["all_features"]
flam_df_clean = flam_df.dropna(subset=ALL_FEATURES + ["log_m200c"])
flam_same = flam_df_clean[flam_df_clean["same_host"]].copy()
flam_diff = flam_df_clean[~flam_df_clean["same_host"]].copy()

from sklearn.model_selection import train_test_split

def get_test_split(df, feats, seed=42, test_size=0.15):
    cols  = feats + ["log_m200c", "host_i", "host_j"]
    valid = df[cols].dropna(subset=feats + ["log_m200c"]).reset_index(drop=True)
    _, te = group_split(valid, test_size=test_size, seed=seed)
    return valid.iloc[te]

DATASETS = {"full": flam_df_clean, "same_host": flam_same, "diff_host": flam_diff}
print("Held-out test sets rebuilt.")

In [ ]:
def evaluate_calibration(posterior, scaler, feats, test_df, n_posterior_samples=200):
    X = scaler.transform(test_df[feats].values.astype(np.float32))
    theta_true = test_df["log_m200c"].values.astype(np.float32)
    with torch.no_grad():
        samples = posterior.sample_batched((n_posterior_samples,),
            x=torch.tensor(X, device=device)).squeeze(-1).cpu().numpy()
    result = calibration_summary(samples, theta_true)
    result["evaluation_rows"] = test_df.index.to_numpy()
    for column in ("host_i", "host_j"):
        if column in test_df:
            result[column] = test_df[column].to_numpy()
    return result

sbc_results = {}
for key, posterior in flam_posteriors.items():
    block, split = key.split("__")
    test_df = get_test_split(DATASETS[split], BLOCKS[block])
    sbc_results[key] = evaluate_calibration(posterior, flam_scalers[key], BLOCKS[block], test_df)
for block, posterior in tng_posteriors.items():
    feats = BLOCKS[block]
    valid = tng_df_clean[feats + ["log_m200c", "host_i", "host_j"]].dropna().reset_index(drop=True)
    # Reuse the exact test membership saved during the corrected fit.
    test_df = valid.iloc[tng_ckpt["split_indices"][block]["test"]]
    sbc_results[f"tng__{block}"] = evaluate_calibration(posterior, tng_scalers[block], feats, test_df)
with open(results_dir / "sbc_results.pkl", "wb") as f:
    pickle.dump(sbc_results, f)

In [ ]:
# The ranks occupy K+1 discrete values, including both endpoints.
KEYS_TO_PLOT = ["core_dynamics__diff_host", "core_morphology__diff_host",
                "tng__core_dynamics", "tng__core_morphology"]
fig, axes = plt.subplots(2, len(KEYS_TO_PLOT), figsize=(18, 8))
for i, key in enumerate(KEYS_TO_PLOT):
    r = sbc_results[key]
    K = r["n_posterior_samples"]
    bins = np.linspace(-.5, K + .5, 21)
    expected, _ = np.histogram(np.arange(K + 1), bins=bins)
    axes[0, i].hist(r["ranks"], bins=bins, alpha=.7)
    axes[0, i].stairs(expected * r["n_test"] / (K + 1), bins,
                      color="k", linestyle="--", label="discrete uniform reference")
    axes[0, i].set_title(f"{key}\nDiscrete CDF distance={r['rank_cdf_distance']:.3f}")
    axes[0, i].set_xlabel("rank")
    axes[0, i].legend(fontsize=7)
    axes[1, i].plot(r["coverage_levels"], r["coverage"], "o-")
    axes[1, i].plot([0, 1], [0, 1], "k--")
    axes[1, i].set(xlabel="nominal central interval", ylabel="empirical coverage",
                   title=f"Median beta={r['median_beta']:.2f} (Gaussian reference: 0.6745)")
plt.tight_layout()
plt.show()

In [ ]:
rows = [{"key": key, **{name: r[name] for name in
    ("median_beta", "rank_cdf_distance", "coverage68", "coverage95", "bias", "n_test")}}
    for key, r in sbc_results.items()]
df_sbc = pd.DataFrame(rows)
print(df_sbc.to_string(index=False))
df_sbc.to_csv(results_dir / "sbc_summary.csv", index=False)
print("Descriptive diagnostics: pairs share hosts; no IID KS p-values or binomial error bars.")